In [ ]:
import os
from typing import Any

import pandas as pd
import numpy as np
import xgboost as xgb
import pandas_ta as ta
from numpy import dtype, ndarray

from utils import download_crypto_data
from datetime import date
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_recall_curve

In [ ]:
download_crypto_data(symbol="BTCUSDT", start=date(2025, 1, 1), end=date(2026, 6, 30), interval="1m", suppress_info=False, tmp_dir="tmp_data")

In [ ]:
train_data = pd.read_csv("data/BTCUSDT-5m-2020-01-01_2024-12-31.csv")
test_data = pd.read_csv("data/BTCUSDT-5m-2025-01-01_2026-06-30.csv")
train_df = pd.DataFrame()
test_df = pd.DataFrame()

Feature Engineering

In [ ]:
def calculate_rsi(price_array: pd.Series, window=14) -> pd.Series:
    delta = price_array.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/window, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1/window, min_periods=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi


def calculate_log_interval_ret(price_array: pd.Series, lag1: int, lag2: int) -> np.ndarray:
    # REQUIRE lag1 < lag2
    return np.log(price_array.shift(lag1) / price_array.shift(lag2))

In [ ]:
t_cost = (0.1 + 0.02 + 0.015) * 0.01  # Transaction cost by Binance + spread + liquidity, or set to zero if wanted
# t_cost = 0

# FEATURE 1: RELATIVE EMA on VWAP
close_window = [10, 20, 50]
for w in close_window:
    train_data["TP"] = (train_data["Open"] + train_data["High"] + train_data["Close"]) / 3
    train_data["VWAP"] = (train_data["TP"] * train_data["Volume"]).rolling(window=w, min_periods=w).sum() / train_data["Volume"].rolling(window=w, min_periods=w).sum()
    train_df[f"FEATURE_RelativeVWAPEMA{w}"] = train_data["VWAP"].ewm(span=w, min_periods=w, adjust=False).mean() / train_data["VWAP"] - 1
    test_data["TP"] = (test_data["Open"] + test_data["High"] + test_data["Close"]) / 3
    test_data["VWAP"] = (test_data["TP"] * test_data["Volume"]).rolling(window=w, min_periods=w).sum() / test_data["Volume"].rolling(window=w, min_periods=w).sum()
    test_df[f"FEATURE_RelativeVWAPEMA{w}"] = test_data["VWAP"].ewm(span=w, min_periods=w, adjust=False).mean() / test_data["VWAP"] - 1

# FEATURE 3: (RELATIVE) MACD + ITS SLOPE
macd_settings = [(10, 20), (10, 50), (20, 50)]
macd_signal = 9
macd_frames_train = [train_df]
macd_frames_test = [test_df]

for fast, slow in macd_settings:
    raw_macd_train = train_data.ta.macd(close='Close', fast=fast, slow=slow, signal=macd_signal).iloc[:, 0]
    rel_macd_train = (raw_macd_train / train_data["Close"]).to_frame()
    col_name = f"FEATURE_MACD_{fast}_{slow}_{macd_signal}"
    rel_macd_train.columns = [col_name]
    slope_col_name = f"{col_name}_slope"
    rel_macd_train_slope = rel_macd_train.ta.slope(close=rel_macd_train[col_name], length=1).to_frame()
    rel_macd_train_slope.columns = [slope_col_name]
    macd_frames_train.extend([rel_macd_train, rel_macd_train_slope])

    raw_macd_test = test_data.ta.macd(close='Close', fast=fast, slow=slow, signal=macd_signal).iloc[:, 0]
    rel_macd_test = (raw_macd_test / test_data["Close"]).to_frame()
    rel_macd_test.columns = [col_name]
    rel_macd_test_slope = rel_macd_test.ta.slope(close=rel_macd_test[col_name], length=1).to_frame()
    rel_macd_test_slope.columns = [slope_col_name]
    macd_frames_test.extend([rel_macd_test, rel_macd_test_slope])

train_df = pd.concat(macd_frames_train, axis=1)
test_df = pd.concat(macd_frames_test, axis=1)

# FEATURE 4: LAGGED RETURNS
ret_window = [[0, 1], [1, 2]]
for w in ret_window:
    lag1, lag2 = w[0], w[1]
    train_df[f"FEATURE_LogReturn{lag1}/{lag2}"] = calculate_log_interval_ret(train_data["Close"], lag1, lag2)
    test_df[f"FEATURE_LogReturn{lag1}/{lag2}"] = calculate_log_interval_ret(test_data["Close"], lag1, lag2)

Random Matrix Theory

In [ ]:
def rmt_clean_features(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Store metadata and center the original data
    columns = df.columns
    index = df.index
    T, N = df.shape
    X_mean = df.mean()
    X_std = df.std()
    X_standardized = (df - X_mean) / X_std
    # 2. Compute empirical correlation matrix & Eigen-decomposition
    corr = X_standardized.corr()
    eigenvalues, eigenvectors = np.linalg.eigh(corr)
    # 3. Calculate Marcenko-Pastur maximum theoretical boundary (lambda_plus)
    lam = N / T
    max_theoretical_lam = (1 + np.sqrt(lam)) ** 2
    # 4. Filter eigenvalues (Clipping Method)
    noise_indices = eigenvalues <= max_theoretical_lam

    if np.any(noise_indices):
        avg_noise_lam = np.mean(eigenvalues[noise_indices])
        clean_eigenvalues = eigenvalues.copy()
        clean_eigenvalues[noise_indices] = avg_noise_lam
        # 5. Reconstruct the denoised correlation matrix
        clean_corr = eigenvectors @ np.diag(clean_eigenvalues) @ eigenvectors.T
        diag = np.diag(clean_corr)
        clean_corr = clean_corr / np.sqrt(np.outer(diag, diag))

        # 6. Map data back to the clean correlation structure
        try:
            emp_lam, emp_vec = np.linalg.eigh(corr)
            emp_lam = np.maximum(emp_lam, 1e-8)
            inv_sqrt_empirical = emp_vec @ np.diag(1.0 / np.sqrt(emp_lam)) @ emp_vec.T

            clean_lam, clean_vec = np.linalg.eigh(clean_corr)
            clean_lam = np.maximum(clean_lam, 0) # drop tiny negative numerical artifacts
            sqrt_clean = clean_vec @ np.diag(np.sqrt(clean_lam)) @ clean_vec.T

            transform_matrix = inv_sqrt_empirical @ sqrt_clean

            X_cleaned = X_standardized.values @ transform_matrix
            df_cleaned = pd.DataFrame(X_cleaned, index=index, columns=columns)

            return (df_cleaned * X_std) + X_mean

        except np.linalg.LinAlgError:
            print("Linear algebra convergence error. Returning original dataframe.")
            return df

    print("No noise eigenvalues detected below the MP boundary. Returning original dataframe.")
    return df

Add target variable

In [ ]:
# train_df = rmt_clean_features(train_df)

# TARGET VARIABLE
train_df[f"TARGET_UpFlag"] = (train_data["Open"] * (1 + t_cost) < train_data["Close"]).shift(-1)
test_df[f"TARGET_UpFlag"] = (test_data["Open"] * (1 + t_cost) < test_data["Close"]).shift(-1)

train_df = train_df.dropna()
test_df = test_df.dropna()
train_df["TARGET_UpFlag"] = train_df["TARGET_UpFlag"].astype(int)
test_df["TARGET_UpFlag"] = test_df["TARGET_UpFlag"].astype(int)

XGBoost Training

In [ ]:
# 1. Separate features (X) and target (y)
target_col = "TARGET_UpFlag"
random_state = 257  # Used when fitting XGB Classifier

# Create a chronological validation set from the tail end of your training data
val_size = int(len(train_df) * 0.25)  # val: validation
train_split_df = train_df.iloc[:-val_size]
val_split_df = train_df.iloc[-val_size:]

X_train = train_split_df.drop(columns=[target_col])
y_train = train_split_df[target_col]

X_val = val_split_df.drop(columns=[target_col])
y_val = val_split_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
neg_pos_ratio = num_neg / num_pos
dampened_neg_pos_ratio = (1 + neg_pos_ratio) / 2
sharpened_neg_pos_ratio = neg_pos_ratio * 2

# 2. Initialize XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=3000,           # Max number of trees
    max_depth=3,                 # Decision rules depth
    learning_rate=0.016,
    subsample=0.65,               # Row (Data rows) subsampling
    colsample_bytree=0.35,        # Col (Features) subsampling
    min_child_weight=120,        # Requires at least 120 samples to create a terminal leaf node
    gamma=3.0,
    random_state=random_state,
    eval_metric='aucpr',
    n_jobs=-1,
    early_stopping_rounds=150,   # Stops fitting when no improvement after consecutive 150 rounds
    scale_pos_weight=neg_pos_ratio,
    reg_alpha=3.0,               # L1 regularization to naturally prune useless features
    reg_lambda=30.0              # L2 regularization to smooth out extreme weight
)

# 3. Train & save the model
# We use the test set as validation to monitor early stopping
print("Training XGBoost model...")
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

os.makedirs("models", exist_ok=True)
model.save_model("models/xgboost_5m_kline.json")

# 4. Evaluate the model
y_pred = model.predict(X_test)
y_val_prob = model.predict_proba(X_val)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]
# best_threshold = 0.8

y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred_custom = (y_test_prob >= best_threshold).astype(int)

print(f"\nDynamically Chosen Best Threshold (Max F1): {best_threshold:.4f}")
print("=== Evaluation Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred_custom):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"\nClassification Report [Threshold={best_threshold:.4f}]:")
print(classification_report(y_test, y_test_pred_custom))

# 5. Check Feature Importance
importance = pd.Series(model.feature_importances_, index=X_train.columns)
print("\n=== Features Importance ===")
print(importance.nlargest(50))